# A Retrieval-Augmented Generation (RAG) and LLM System for Context-Aware, Multimodal Study Assistance

**Federal University Dutsin-Ma (FUDMA) — Final Year Project**

By Sanusi Shafii

s.shafii27683@fudutsinma.edu.ng

---

This notebook implements a study assistant that:
- Accepts one or more PDF documents (text-based or scanned)
- Extracts text using PyMuPDF and falls back to Tesseract OCR for image-based pages
- Breaks the text into chunks and stores them in a FAISS vector index
- Retrieves the most relevant chunks for any question you ask
- Sends the question and retrieved context to Google Gemini to generate a grounded answer

**Run each cell from top to bottom. Do not skip any cell.**

## Cell 1 — Install Required Libraries

This installs all the Python packages and system tools needed for the project.
Run this cell once when you open the notebook in Colab.

In [ ]:
!pip install -q pymupdf pytesseract sentence-transformers faiss-cpu groq pillow ipywidgets
!apt-get install -q tesseract-ocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.5 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


## Cell 2 — Import Libraries

All the libraries used in this project are imported here in one place.

In [ ]:
import os
import numpy as np
from PIL import Image

import fitz                              # PyMuPDF — for reading PDF files
import pytesseract                       # Tesseract OCR — for scanned/image pages
import faiss                             # FAISS — for fast vector similarity search

from sentence_transformers import SentenceTransformer   # For converting text to embeddings
import google.generativeai as genai                     # Google Gemini LLM API
from google.colab import files                          # Colab file upload utility

import ipywidgets as widgets             # For building the interactive interface
from IPython.display import display, HTML, clear_output

print("All libraries imported successfully.")

All libraries imported successfully.


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## Cell 3 — Configuration

Set your Gemini API key here. You can get a free API key from:
https://aistudio.google.com/app/apikey

The other settings control how text is chunked and how many results are retrieved.

In [ ]:
import os
import numpy as np
from PIL import Image

import fitz                              # PyMuPDF — for reading PDF files
import pytesseract                       # Tesseract OCR — for scanned/image pages
import faiss                             # FAISS — for fast vector similarity search

from sentence_transformers import SentenceTransformer   # For converting text to embeddings
import google.generativeai as genai                     # Google Gemini LLM API
from google.colab import files                          # Colab file upload utility

import ipywidgets as widgets             # For building the interactive interface
from IPython.display import display, HTML, clear_output

print("All libraries imported successfully.")
# --- Configuration ---
from groq import Groq

GROQ_API_KEY = "UseYourGroqAPIKey"

CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
TOP_K         = 4

# Initialise the Groq client
groq_client = Groq(api_key=GROQ_API_KEY)

# Load the sentence embedding model (downloads on first run)
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Configuration complete.")

All libraries imported successfully.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Configuration complete.


## Cell 4 — Document Processing Functions

These functions handle reading PDF documents and splitting text into manageable chunks.

- `extract_text_from_pdf`: reads each page; if a page has no selectable text (scanned), it uses Tesseract OCR.
- `chunk_text`: splits a long piece of text into smaller overlapping chunks.

In [ ]:
def extract_text_from_pdf(pdf_path):
    """
    Extract all text from a PDF file.

    For pages that contain selectable text, PyMuPDF is used directly.
    For pages that are images (scanned documents), Tesseract OCR is applied.

    Parameters
    ----------
    pdf_path : str
        Path to the PDF file on disk.

    Returns
    -------
    str
        The combined extracted text from all pages.
    """
    document = fitz.open(pdf_path)
    full_text = ""

    for page_index, page in enumerate(document):
        text = page.get_text()

        if text.strip():
            # Page has selectable text — use it directly
            full_text += text
        else:
            # Page is an image — render it and run OCR
            pixel_map = page.get_pixmap(dpi=200)
            image = Image.frombytes("RGB", [pixel_map.width, pixel_map.height], pixel_map.samples)
            ocr_text = pytesseract.image_to_string(image)
            full_text += ocr_text

    document.close()
    return full_text


def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Split a long string of text into smaller overlapping chunks.

    Overlapping ensures that sentences near chunk boundaries are not lost.

    Parameters
    ----------
    text : str
        The full document text to split.
    chunk_size : int
        Maximum number of characters per chunk.
    overlap : int
        Number of characters to repeat between consecutive chunks.

    Returns
    -------
    list of str
        A list of text chunks.
    """
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap

    return chunks


print("Document processing functions defined.")

Document processing functions defined.


## Cell 5 — Embedding and Index Functions

These functions convert text chunks into numerical vectors (embeddings) and store them in a FAISS index.
FAISS enables fast similarity search — given a question, it finds the most related chunks.

In [ ]:
# These two variables hold the FAISS index and the original text chunks globally
chunk_store  = []
faiss_index  = None


def build_faiss_index(chunks):
    """
    Embed a list of text chunks and store them in a FAISS index.

    Each chunk is converted into a fixed-size vector using the
    sentence embedding model. FAISS uses these vectors to find
    the closest matches for any new query.

    Parameters
    ----------
    chunks : list of str
        The text chunks to embed and index.
    """
    global faiss_index, chunk_store

    chunk_store = chunks
    print("Embedding chunks — this may take a moment...")

    embeddings = embedding_model.encode(chunks, show_progress_bar=True)
    embeddings = np.array(embeddings).astype("float32")

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    faiss_index = index
    print(f"FAISS index built with {len(chunks)} chunks.")


def retrieve_relevant_chunks(query, top_k=TOP_K):
    """
    Find and return the text chunks most relevant to a given query.

    The query is embedded and compared to all stored chunk embeddings.
    The chunks with the smallest distance (most similar) are returned.

    Parameters
    ----------
    query : str
        The user's question.
    top_k : int
        Number of chunks to return.

    Returns
    -------
    list of str
        The most relevant text chunks.
    """
    if faiss_index is None:
        return []

    query_vector = embedding_model.encode([query]).astype("float32")
    distances, indices = faiss_index.search(query_vector, top_k)

    return [chunk_store[i] for i in indices[0] if i < len(chunk_store)]


print("Embedding and retrieval functions defined.")

Embedding and retrieval functions defined.


## Cell 6 — Load and Process Documents

Run this cell to upload your PDF study materials. You can upload more than one file.
The system will extract the text, split it into chunks, and build the search index.

In [ ]:
def load_documents():
    """
    Prompt the user to upload PDF files, then extract text and build the FAISS index.

    This function coordinates the full document ingestion pipeline:
    upload -> extract -> chunk -> embed -> index.
    """
    print("Please select and upload your PDF document(s).")
    uploaded_files = files.upload()

    if not uploaded_files:
        print("No files were uploaded. Please run this cell again.")
        return

    all_chunks = []

    for filename, file_bytes in uploaded_files.items():
        # Save the uploaded file to disk so PyMuPDF can read it
        with open(filename, "wb") as f:
            f.write(file_bytes)

        print(f"\nProcessing: {filename}")
        extracted_text = extract_text_from_pdf(filename)

        if not extracted_text.strip():
            print(f"  Warning: No text could be extracted from {filename}. Skipping.")
            continue

        chunks = chunk_text(extracted_text)
        all_chunks.extend(chunks)
        print(f"  {len(chunks)} chunks extracted from {filename}.")

    if not all_chunks:
        print("\nNo content was extracted from any of the uploaded files.")
        return

    print(f"\nTotal chunks across all documents: {len(all_chunks)}")
    build_faiss_index(all_chunks)
    print("\nDocuments loaded. You may now scroll down to the Study Assistant.")


load_documents()

Please select and upload your PDF document(s).


Saving SPECIAL TOPICS IN ICT.docx to SPECIAL TOPICS IN ICT.docx

Processing: SPECIAL TOPICS IN ICT.docx
  214 chunks extracted from SPECIAL TOPICS IN ICT.docx.

Total chunks across all documents: 214
Embedding chunks — this may take a moment...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

FAISS index built with 214 chunks.

Documents loaded. You may now scroll down to the Study Assistant.


## Cell 7 — Query Function

This function ties the retrieval step and the LLM together.
It takes your question, finds the most relevant chunks, and asks Gemini to answer based only on that context.

In [ ]:
def answer_question(question):
    """
    Answer a study question using retrieved document context and the Groq LLM.

    Retrieves the most relevant text chunks from the FAISS index, constructs
    a prompt with those chunks as context, and sends it to Groq for a
    grounded, context-aware response.

    Parameters
    ----------
    question : str
        The user's study question.

    Returns
    -------
    str
        The LLM-generated answer, or an error message if the index is not ready.
    """
    if faiss_index is None:
        return "The document index is not ready. Please run Cell 6 first to upload your documents."

    if not question.strip():
        return "Please enter a question."

    relevant_chunks = retrieve_relevant_chunks(question)

    if not relevant_chunks:
        return "No relevant content was found in the uploaded documents for this question."

    context = "\n\n".join(relevant_chunks)

    prompt = (
        "You are a helpful and accurate study assistant.\n"
        "Use only the context provided below to answer the question.\n"
        "If the context does not contain enough information, say so clearly.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1024
    )

    return response.choices[0].message.content


print("Query function defined.")

Query function defined.


## Cell 8 — Interactive Study Assistant

This is the main interface. Type your question in the box and click **Ask** to get an answer.
The assistant will search your uploaded documents and use Gemini to generate a response.

Make sure you have run all cells above before using this interface.

In [ ]:
# --- Interactive Study Assistant Interface ---

# Input box for the user's question
question_box = widgets.Textarea(
    placeholder="Type your study question here and click Ask...",
    layout=widgets.Layout(width="100%", height="90px")
)

# Button to submit the question
ask_button = widgets.Button(
    description="Ask",
    button_style="primary",
    layout=widgets.Layout(width="130px", height="38px")
)

# Button to clear the current output
clear_button = widgets.Button(
    description="Clear",
    button_style="warning",
    layout=widgets.Layout(width="130px", height="38px")
)

# Area where the answer will be displayed
output_area = widgets.Output()


def on_ask_clicked(button):
    """
    Handle the Ask button click event.

    Reads the question from the input box, calls the answer_question function,
    and displays the result inside the output area as a styled HTML card.
    """
    question = question_box.value.strip()

    output_area.clear_output()

    if not question:
        with output_area:
            print("Please type a question before clicking Ask.")
        return

    with output_area:
        print("Searching documents and generating answer...")

    answer = answer_question(question)

    output_area.clear_output()

    with output_area:
        display(HTML(
            "<div style='"
            "border: 1px solid #d0d0d0;"
            "border-radius: 8px;"
            "padding: 18px;"
            "margin-top: 12px;"
            "background-color: #f8f8f8;"
            "font-family: Arial, sans-serif;"
            "font-size: 14px;"
            "line-height: 1.7;"
            "'>"
            "<p style='font-weight: bold; font-size: 15px; margin-bottom: 6px; color: #222;'>Question</p>"
            f"<p style='color: #444; margin-bottom: 14px;'>{question}</p>"
            "<hr style='border: none; border-top: 1px solid #ddd; margin-bottom: 14px;'>"
            "<p style='font-weight: bold; font-size: 15px; margin-bottom: 6px; color: #222;'>Answer</p>"
            f"<p style='color: #333;'>{answer}</p>"
            "</div>"
        ))


def on_clear_clicked(button):
    """
    Handle the Clear button click event.

    Clears the question input and the output display area.
    """
    question_box.value = ""
    output_area.clear_output()


# Attach the handler functions to the buttons
ask_button.on_click(on_ask_clicked)
clear_button.on_click(on_clear_clicked)

# Arrange the buttons side by side
button_row = widgets.HBox([ask_button, clear_button])

# Display the full interface
display(HTML(
    "<div style='font-family: Arial, sans-serif; margin-bottom: 10px;'>"
    "<h3 style='margin-bottom: 4px;'>RAG Study Assistant</h3>"
    "<p style='color: #555; font-size: 13px;'>Ask any question based on the documents you uploaded.</p>"
    "</div>"
))
display(question_box, button_row, output_area)

Textarea(value='', layout=Layout(height='90px', width='100%'), placeholder='Type your study question here and …

Output()